<div align="center">

# Use fse in BERTopic and KeyBERT

Find the topics of a news collection and the key phrases of one article, with fse as the encoder.

</div>

## When do you need this?

[BERTopic](https://github.com/MaartenGr/BERTopic) groups many documents into **topics** and names each topic with words.
[KeyBERT](https://github.com/MaartenGr/KeyBERT) picks the **key phrases** of one document.
Both turn texts into vectors first. Both let you plug in your own embedder, so fse can supply the vectors without any change to either project.

| Step | What happens | Call |
|---|---|---|
| 1 | Load a pinned static model into an fse encoder | `fse.Encoder.from_pretrained` |
| 2 | Wrap the encoder for each library with one small class | `embed(documents, verbose=False)` |
| 3 | Find topics in 3,000 news articles | `BERTopic(embedding_model=...)` |
| 4 | Assign new articles to the topics, then save and reload the model | `transform`, `save`, `BERTopic.load` |
| 5 | Extract key phrases from single articles | `KeyBERT(model=...)` |
| 6 | Compare with the Model2Vec backend that each library ships | `Model2VecBackend`, `StaticModel` |

**This recipe is a correctness check, not a benchmark.** It makes no speed claim and no claim about topic or keyword quality.
Supported versions: **BERTopic 0.17.4 and KeyBERT 0.9.0**. BERTopic and KeyBERT support Python 3.10 and newer and fse needs 3.11 or newer, so the combination needs 3.11 or newer. The shared recipe setup pins packages that need Python 3.12 or newer and documents Python 3.13, so use 3.13 with it.
The [shared setup](README.md#run-a-recipe) installs the pinned packages, including `bertopic` and `keybert` from `requirements.txt`.

## 1. Texts and an fse encoder

We use the first 3,000 articles of the AG News test split to find topics. The next 500 articles are new texts for later.
The data is pinned in [_helpers.py](_helpers.py).
The model is [POTION-base-8M](https://huggingface.co/minishlab/potion-base-8M) at a pinned revision. It is English-only.
`recipe.encoder()` loads it with `fse.Encoder.from_pretrained`, with the settings stored in the model (including unit-length vectors).

In [1]:
from _helpers import Recipe
import numpy as np
import bertopic
import keybert

recipe = Recipe()
texts, labels = recipe.dataset("AG News", "test")
encoder = recipe.encoder()
docs, new_docs = texts[:3000], texts[3000:3500]
print("Documents:", len(docs), "New documents:", len(new_docs))
print("BERTopic", bertopic.__version__, "KeyBERT", keybert.__version__)

W1116 10:17:26.174000 74730 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Recipe run: 2026-11-16. Generated files are temporary.
Apple M1 Pro. macOS-27.0.1-arm64-arm-64bit-Mach-O. Python 3.13.16.


test.jsonl: reconstructing file:   0%|          |  0.00B / 2.13MB            

test.jsonl: downloading bytes:           |  0.00B            

Documents: 3000 New documents: 500
BERTopic 0.17.4 KeyBERT 0.9.0


## 2. One small class for each library

Both libraries define a `BaseEmbedder` class and recognize an embedder only when it is a subclass of **their own** `BaseEmbedder`.
The two base classes are separate and not interchangeable. KeyBERT does not recognize a BERTopic embedder and silently falls back to its default sentence-transformers model, which we found by testing, not in the documentation.

Each library needs only one method, `embed(documents, verbose=False)`, which returns a 2-D array with one row per text.
BERTopic also calls `embed_words` and `embed_documents`. Their default in the base class is `embed`, so we do not override them.
The base class stores the encoder as `self.embedding_model`. The shared part is a small mixin, and each library gets a one-line subclass.

In [2]:
from bertopic.backend import BaseEmbedder as BERTopicBase
from keybert.backend import BaseEmbedder as KeyBERTBase


class FseEmbed:
    """Turn texts into vectors with an fse encoder. Put this class first in the bases."""

    def embed(self, documents, verbose=False):
        if isinstance(documents, str):
            documents = [documents]
        return self.embedding_model.encode(list(documents))


class FseForBERTopic(FseEmbed, BERTopicBase):
    pass


class FseForKeyBERT(FseEmbed, KeyBERTBase):
    pass


topic_embedder = FseForBERTopic(encoder)
vectors = topic_embedder.embed(docs[:3])
print("embed returns:", type(vectors).__name__, vectors.dtype, vectors.shape)

embed returns: ndarray float32 (3, 256)


## 3. Find topics with BERTopic

BERTopic embeds the documents, reduces the vectors with UMAP, clusters them with HDBSCAN, and describes each cluster with words.
UMAP and HDBSCAN are BERTopic's usual parts. We fix `random_state` so a rerun on the same machine gives the same result. We use `hdbscan.HDBSCAN` with `prediction_data=True` because `BERTopic.transform` needs a clusterer that can predict.
Topic `-1` collects the documents that belong to no cluster. These are demonstration settings, so **read the topic words** before you trust a topic.
We wrap the embedder in a counter to see which methods BERTopic calls.

In [3]:
from bertopic import BERTopic
from bertopic.representation import MaximalMarginalRelevance
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP

calls = []


class CountingEmbedder(FseForBERTopic):
    def embed(self, documents, verbose=False):
        calls.append(len(documents))
        return super().embed(documents, verbose)


def build(embedder, representation=None):
    return BERTopic(
        embedding_model=embedder,
        umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=7),
        hdbscan_model=HDBSCAN(min_cluster_size=30, prediction_data=True),
        vectorizer_model=CountVectorizer(stop_words="english"),
        representation_model=representation,
        calculate_probabilities=False,
    )


topic_model = build(CountingEmbedder(encoder))
topics, _ = topic_model.fit_transform(docs)
assert len(topics) == len(docs)
print("Topics found:", len(set(topics) - {-1}), "Documents in no topic:", topics.count(-1))
print("embed calls (texts per call):", calls)
info = topic_model.get_topic_info()[["Topic", "Count", "Name"]]
print(info.head(10).to_string(index=False))

Topics found: 15 Documents in no topic: 549
embed calls (texts per call): [3000]
 Topic  Count                              Name
    -1    549             -1_said_39_search_new
     0    812                0_39_win_athens_ap
     1    425  1_microsoft_new_software_windows
     2    264         2_iraq_baghdad_iraqi_said
     3    192    3_stocks_prices_profit_percent
     4    147        4_bush_kerry_korea_nuclear
     5     99           5_million_fund_court_39
     6     81 6_airways_boeing_airline_airlines
     7     74  7_hurricane_ivan_florida_frances
     8     69           8_space_nasa_earth_mars


BERTopic called `embed` once with all 3,000 documents. The topic words are chosen from the vocabulary of the documents, not from the vectors, so this step needs no word vectors.
**Word vectors are needed** only when you ask BERTopic for a representation that compares words with topics. `MaximalMarginalRelevance` does that.
The embedder then has to embed single words as well as documents, which it does through the same `embed` method.

In [4]:
calls.clear()
mmr_model = build(CountingEmbedder(encoder), MaximalMarginalRelevance(diversity=0.3))
mmr_topics, _ = mmr_model.fit_transform(docs)
print("embed calls (texts per call):", calls[:1], "then", len(calls) - 1, "calls for words; the largest has", max(calls[1:]), "words")
print("Same topic assignment as before:", mmr_topics == topics)
print(mmr_model.get_topic_info()[["Topic", "Count", "Name"]].head(5).to_string(index=False))

embed calls (texts per call): [3000] then 32 calls for words; the largest has 30 words
Same topic assignment as before: True
 Topic  Count                            Name
    -1    549    -1_google_reuters_china_says
     0    812         0_olympic_game_team_cup
     1    425  1_microsoft_software_linux_ibm
     2    264     2_iraq_killed_police_cleric
     3    192 3_stocks_percent_reuters_dollar


## 4. Assign new documents, save, reload

`transform` embeds new documents with the same embedder and assigns each to a topic.
`save` stores the topic model **without** the embedder when we pass `save_embedding_model=False`. We choose this on purpose: BERTopic stores an embedder only as a model name (a string), and an fse encoder has none.
So you pass an embedder again when you reload, with `BERTopic.load(path, embedding_model=...)`. If you do not pass one, a loaded model has no embedder and cannot embed new texts.
The safetensors format keeps the topic vectors and the topic words. It does not keep UMAP and HDBSCAN, so a loaded model assigns a document to the topic with the closest topic vector.
That is a different rule from the one of the fitted model, so we compare both on the 500 new documents and report what we see.

In [5]:
new_topics, _ = topic_model.transform(new_docs)
folder = recipe.work / "topic-model"
topic_model.save(folder, serialization="safetensors", save_ctfidf=True, save_embedding_model=False)
loaded = BERTopic.load(folder, embedding_model=FseForBERTopic(encoder))
loaded_topics, _ = loaded.transform(new_docs)

print("Loaded model has the same topic words:", loaded.get_topic_info()["Name"].tolist() == info["Name"].tolist() or "differs")
print("New documents with the same topic in the fitted and the loaded model:", int(np.mean(np.array(new_topics) == np.array(loaded_topics)) * 100), "%")
print("New documents in no topic (fitted model):", list(new_topics).count(-1), "of", len(new_docs))

2026-11-16 10:17:46,769 - BERTopic - WARNING: You are saving a BERTopic model without explicitly defining an embedding model.If you are using a sentence-transformers model or a HuggingFace model supportedby sentence-transformers, please save the model by using a pointer towards that model.For example, `save_embedding_model='sentence-transformers/all-mpnet-base-v2'`


Loaded model has the same topic words: True
New documents with the same topic in the fitted and the loaded model: 72 %
New documents in no topic (fitted model): 142 of 500


## 5. Extract key phrases with KeyBERT

KeyBERT embeds the document, embeds every candidate word or phrase, and returns the phrases whose vectors are closest to the document vector.
Both calls go through `embed`. Pass the embedder with `KeyBERT(model=...)`.
We use phrases of one or two words and show the first three articles.

In [6]:
kw_model = keybert.KeyBERT(model=FseForKeyBERT(encoder))
for text in texts[:3]:
    print(" ".join(text.split())[:110])
    print("  ", kw_model.extract_keywords(text, keyphrase_ngram_range=(1, 2), stop_words="english", top_n=4))

Fears for T N pension after talks Unions representing workers at Turner Newall say they are 'disappointed' aft
   [('talks unions', 0.5841), ('pension talks', 0.5541), ('disappointed talks', 0.4713), ('fears pension', 0.4642)]
The Race is On: Second Private Team Sets Launch Date for Human Spaceflight (SPACE.com) SPACE.com - TORONTO, Ca
   [('rocketeers competing', 0.558), ('spaceflight space', 0.5534), ('human spaceflight', 0.5419), ('launch', 0.539)]
Ky. Company Wins Grant to Study Peptides (AP) AP - A company founded by a chemistry researcher at the Universi
   [('peptides ap', 0.6857), ('study peptides', 0.6807), ('peptides', 0.6192), ('better peptides', 0.6034)]


KeyBERT also accepts a list of documents and returns one list of phrases per document.
`use_mmr=True` and `use_maxsum=True` call the same `embed` method, so they need nothing more from the embedder.

In [7]:
batch = kw_model.extract_keywords(texts[:3], keyphrase_ngram_range=(1, 2), stop_words="english", top_n=3, use_mmr=True, diversity=0.5)
assert len(batch) == 3
for keywords in batch:
    print([phrase for phrase, _ in keywords])

['talks unions', 'federal mogul', 'stricken parent']
['rocketeers competing', 'launch date', 'com space']
['peptides ap', 'founded chemistry', 'grant study']


## 6. Compare with the Model2Vec backend

Both libraries ship a `Model2VecBackend`. We load the same pinned POTION-base-8M files with `model2vec.StaticModel` and run the same steps.
We report what we observe. We make no claim about which is better, because we did not measure topic or keyword quality.

In [8]:
from bertopic.backend import Model2VecBackend
from model2vec import StaticModel
from sklearn.metrics import adjusted_rand_score

static = StaticModel.from_pretrained(str(recipe.model_folder()))
ours, theirs = topic_embedder.embed(docs), Model2VecBackend(static).embed(docs)
print("Largest vector difference:", float(np.abs(ours - theirs).max()))

baseline = build(Model2VecBackend(static))
baseline_topics, _ = baseline.fit_transform(docs)
print("fse encoder:      topics", len(set(topics) - {-1}), "no topic", topics.count(-1))
print("Model2VecBackend: topics", len(set(baseline_topics) - {-1}), "no topic", baseline_topics.count(-1))
print("Adjusted Rand index between both assignments:", round(adjusted_rand_score(topics, baseline_topics), 3))
print("Documents with the same topic number:", int(np.mean(np.array(topics) == np.array(baseline_topics)) * 100), "%")
print("Topic words, fse encoder:     ", [w for w, _ in topic_model.get_topic(0)][:5])
print("Topic words, Model2VecBackend:", [w for w, _ in baseline.get_topic(0)][:5])

same = 0
for text in texts[:20]:
    a = kw_model.extract_keywords(text, keyphrase_ngram_range=(1, 2), stop_words="english", top_n=5)
    b = keybert.KeyBERT(model=static).extract_keywords(text, keyphrase_ngram_range=(1, 2), stop_words="english", top_n=5)
    same += a == b
print("KeyBERT: 20 articles, identical phrases and scores for", same)

Largest vector difference: 5.960464477539063e-08


fse encoder:      topics 15 no topic 549
Model2VecBackend: topics 17 no topic 513
Adjusted Rand index between both assignments: 0.911
Documents with the same topic number: 75 %
Topic words, fse encoder:      ['39', 'win', 'athens', 'ap', 'season']
Topic words, Model2VecBackend: ['39', 'win', 'athens', 'ap', 'season']
KeyBERT: 20 articles, identical phrases and scores for 20


## What the results show
In the run recorded in this notebook:

- BERTopic accepted the fse embedder, found 15 topics in 3,000 articles, and called `embed` for documents and, with `MaximalMarginalRelevance`, for words.
- A reloaded model needs the embedder again. It assigned 72% of 500 new articles to the same topic as the fitted model. This is a different assignment rule, not an error.
- KeyBERT returned the same phrases and scores as its own Model2Vec backend for all 20 articles we compared.
- BERTopic's topics differ between the fse encoder and the Model2Vec backend, although the vectors differ by at most 6e-8 (float rounding). The adjusted Rand index between both assignments is 0.911, where 1 means identical. The topic counts are 15 and 17.
  A probable reason is that UMAP amplifies tiny differences, but we did not test it. The numbers are one run on one machine and show no better or worse quality.

## Limits and next steps

- **Versions.** Tested with BERTopic 0.17.4, KeyBERT 0.9.0 and fse on Python 3.13. Both libraries support Python 3.10, fse needs 3.11 or newer.
  Another version may change the `BaseEmbedder` contract, the save format or the defaults. This recipe checks no other version. BERTopic 1.0 is in pre-release and was not tested.
- **Two base classes.** BERTopic and KeyBERT each check for their own `BaseEmbedder`. A class for one is not recognized by the other.
- **Model.** POTION-base-8M is a small English model. A static model gives every word one vector regardless of context, so it cannot tell the two meanings of "bank" apart, and a topic model built on it groups by wording more than by meaning.
  Use a model that fits your language and domain. Topic quality and keyword quality were **not** measured here.
- **Stable topics.** UMAP amplifies tiny float differences, so a rerun on another machine or library version can give other topic numbers. Fixing `random_state` helps on one machine only.
- **Save and load.** Pass an embedder again when you load a BERTopic model. The pickle format is not tested here, because it would store the embedder object with it.
- **Heavy installation.** BERTopic installs sentence-transformers and PyTorch even when you use fse for the vectors.
- **No quality claim and no speed claim.** We labeled no topics or keywords and measured no time.
- **No glue package.** This recipe uses public fse names only. There is no fse adapter for BERTopic or KeyBERT.

- [fse 101](00-fse-101.ipynb)
- [Integrations](06-integrations.ipynb), with a shorter BERTopic section that passes ready-made vectors.
- [Near-duplicate removal with SemHash](14-semhash-dedup.ipynb)
- [Recipe index](README.md)

In [9]:
recipe.close()